In [8]:
import random
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

csv_file = "./train_data.csv"
data = pd.read_csv(csv_file)
data

,text,label
0,How much cash does each player begin with in M...,0
1,What is the starting money for each player in ...,0
2,How much money do players receive at the start...,0
3,What amount of cash does every player start wi...,0
4,How much starting capital does each Monopoly p...,0
...,...,...
401,How do you choose the right shoes for walking,1
402,What is the purpose of a seatbelt,1
403,How do you make a simple craft with cardboard,1
404,What are the rules of a spelling bee,1


In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_length = 128 

# Tokenize and prepare data
tokenized_texts = []
for index, row in data.iterrows():
    text = str(row['text'])
    label = int(row['label'])
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    tokenized_texts.append({
        'input_ids': inputs['input_ids'].flatten(),
        'attention_mask': inputs['attention_mask'].flatten(),
        'label': torch.tensor(label, dtype=torch.long)
    })

In [10]:
tokenized_texts[0]

{'input_ids': tensor([  101,  2129,  2172,  5356,  2515,  2169,  2447,  4088,  2007,  1999,
         15404,  1029,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [11]:
# Split into training and test sets
train_data, test_data = train_test_split(tokenized_texts, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Define optimizer and criterion
optimizer = AdamW(model.parameters(), lr=2e-5)

# Define class weights
class_counts = [len(data[data['label']==0]), len(data[data['label']==1])]  
total_samples = sum(class_counts)
class_weights = [total_samples / (2.0 * count) for count in class_counts]

# Convert class weights to a PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

criterion = torch.nn.CrossEntropyLoss()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9028.99it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [12]:
# Training loop
def train(model, optimizer, train_loader, criterion, num_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        print(epoch)
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["label"]

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {total_loss/len(train_loader)}")

# Train the model
train(model, optimizer, train_loader, criterion)

0
Epoch 1/3, Training Loss: 0.43812783488205503
1
Epoch 2/3, Training Loss: 0.13355349092966035
2
Epoch 3/3, Training Loss: 0.038556272784868874


In [13]:
# Save the fine-tuned model
model_save_path = "./fine_tuned_bert_model.pth"

torch.save(model.state_dict(), model_save_path)
print(f"Fine-tuned BERT model saved to '{model_save_path}'")

Fine-tuned BERT model saved to './fine_tuned_bert_model.pth'


In [14]:
#Updated evaluate function
def evaluate(model, test_loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["label"]

            outputs = model(input_ids, attention_mask=attention_mask)
            _, predicted = torch.max(outputs.logits, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # Calculate precision, recall, and f1-score
    print(classification_report(y_true, y_pred, target_names=["Class 0", "Class 1"]))
    return [y_true, y_pred]


In [15]:
# Evaluate the model with adjusted threshold
r = evaluate(model, test_loader)

              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        47
     Class 1       1.00      1.00      1.00        35

    accuracy                           1.00        82
   macro avg       1.00      1.00      1.00        82
weighted avg       1.00      1.00      1.00        82



In [18]:
from transformers import AutoModelForSequenceClassification

model_name = "bert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
state_dict = torch.load("./fine_tuned_bert_model.pth", map_location="cpu")
model.load_state_dict(state_dict)

model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8092.56it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [84]:
texts = "monopoly is a good game"

inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():
    outputs = model(**inputs)

preds = torch.argmax(outputs.logits, dim=1)

print(preds.tolist()[0])

0


In [34]:
# save local
model.save_pretrained("./discord_bert_model")
tokenizer.save_pretrained("./discord_bert_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


('./discord_bert_model\\tokenizer_config.json',
 './discord_bert_model\\tokenizer.json')

In [79]:
import os
from dotenv import load_dotenv

load_dotenv()

token_HF = os.getenv("HUGGINGFACE_TOKEN")

model.push_to_hub(F"Equi00/discord-rag-chatbot", token=token_HF)
tokenizer.push_to_hub(f"Equi00/discord-rag-chatbot", token=token_HF)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]
Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 2.88MB/s  
New Data Upload: 100%|██████████|  100MB /  100MB, 2.88MB/s  
c:\Users\Usuario\Desktop\Discord-RAG-Chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--EQUI00--discord-rag-chatbot. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https:

CommitInfo(commit_url='https://huggingface.co/Equi00/discord-rag-chatbot/commit/eb32f4453c536866dc6b45af1199c900e5fc30a5', commit_message='Upload tokenizer', commit_description='', oid='eb32f4453c536866dc6b45af1199c900e5fc30a5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Equi00/discord-rag-chatbot', endpoint='https://huggingface.co', repo_type='model', repo_id='Equi00/discord-rag-chatbot'), pr_revision=None, pr_num=None)

In [82]:
from transformers import AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(f"Equi00/discord-rag-chatbot")
tokenizer = AutoTokenizer.from_pretrained(f"Equi00/discord-rag-chatbot")

model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7902.65it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,